# Interactive Peak Fitting Tool for FIB TOF-SIMS / Mass Spectra

## Overview

This Jupyter Notebook-based tool is designed for interactive peak fitting of FIB TOF-SIMS and other mass spectrometry data. It supports Gaussian, Lorentzian, and PseudoVoigt models with optional baseline correction, smoothing, and default settings on uranium isotope ratio analysis.

**Current Version:** 1.6.2 (Included 234U peaks)

**Author:** Xiao Sun [https://github.com/xiaosun622]

---

## Workflow: Step-by-Step Procedure

### 1. Load Spectrum File

* Reads a tab-delimited `.txt` file.
* Expected columns:

  * `mass/charge (m/Q)` (x-axis)
  * `Total (cts/TOF-Extraction)` (intensity)

### 2. Apply Smoothing (Optional)

* Savitzky–Golay smoothing filter reduces noise while preserving peak shape.
* Adjustable parameters:

  * `Smooth Win`: Window size
  * `Polyorder`: Polynomial order used in smoothing

### 3. Define Peak Regions

* User inputs peak center and range width (±) for each ion.
* Regions are defined as: `center ± range_width`

### 4. Baseline Correction (if enabled)

* Options:

  * `average`: Flat baseline using predicted intensity ± offset
  * `linear`: Line fit to surrounding regions
  * `polynomial`: 2nd-order polynomial fit
* This background is subtracted from the signal to isolate the peak.

### 5. Select Fit Model

* Choose between symmetric or asymmetric peak shapes:

  * Gaussian
  * Lorentzian
  * PseudoVoigt (or Voigt for asymmetric cases)

### 6. Fit the Peak

* The fitting is applied to the **baseline-corrected signal**.
* Initial parameters are estimated (center, amplitude, sigma).
* The model is optimized to minimize the squared difference between corrected data and prediction.

### 7. Extract Results

* From the fit result:

  * Best-fit curve (`model_prediction`)
  * Fitting Deviation (`data - prediction`)
  * R² (goodness of fit)
  * Area (peak amplitude)

### 8. Plot Outputs

Each subplot includes:

* Smoothed signal
* Corrected signal
* Model fit
* **Fitting Deviation** (formerly “Residuals”)

### 9. Calculate Isotope Ratios

For isotopes (e.g. 234U, 235U, 238U), the ratio is:

```math
\text{Ratio} = \frac{\text{Area}_{235}}{\text{Area}_{234} + \text{Area}_{235} + \text{Area}_{238}}
```

### 10. Save Outputs (Manual Trigger)

* Press "Save Results" after fitting to export:

  * A PNG image of all plots
  * A CSV summary table with areas, R², and isotope ratios

---

## Terminology

| Term                  | Meaning                                                         |
| --------------------- | --------------------------------------------------------------- |
| **Fitting Deviation** | Difference between corrected data and model fit                 |
| **Baseline**          | Estimated background under the peak (subtracted before fitting) |
| **Best Fit**          | Model prediction using optimized parameters                     |
| **R²**                | Coefficient of determination; closer to 1 means better fit      |

---

## Requirements

* Python 3.7+
* `pandas`, `numpy`, `matplotlib`, `scipy`, `lmfit`, `ipywidgets`

Install via pip:

```bash
pip install pandas numpy matplotlib scipy lmfit ipywidgets
```

---

MIT License

Copyright (c) 2025 xiaosun622

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [10]:
%pip install pandas numpy matplotlib scipy lmfit ipywidgets

In [11]:
# === Interactive Peak Fitting Script for TOF-SIMS / Mass Spectra ===
"""
Title: Interactive Peak Fitting Tool
Author: Xiao Sun [https://github.com/xiaosun622]
Version: 1.6.2 (Inclusded 234U peaks)
Date: 10-12-2025

Description:
This script provides an interactive Jupyter Notebook-based interface for peak fitting in TOF-SIMS or general mass spectrometry spectra.
It enables users to upload a tab-delimited spectrum file, smooth data, select fit models, perform baseline correction, and calculate isotope ratios.
"""

# === [IMPORTS] ===
import os, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from lmfit.models import GaussianModel, LorentzianModel, PseudoVoigtModel, VoigtModel
import ipywidgets as widgets
from IPython.display import display, clear_output

# Try importing skewed Gaussian model for asymmetric peak fitting
try:
    from lmfit.models import SkewedGaussianModel
except ImportError:
    SkewedGaussianModel = None

# Suppress warning messages that are not critical
warnings.filterwarnings("ignore", category=UserWarning, module="uncertainties.core")

# === [DATA INIT] ===
x_column = 'mass/charge (m/Q)'
y_column = 'Total (cts/TOF-Extraction)'

data = None
x = None
y = None
fig = None
df_summary = None
results_last_filename = None

# === [WIDGETS] ===
upload_widget = widgets.FileUpload(
    accept='.txt',
    multiple=False,
    description='Upload .txt',
    layout=widgets.Layout(width='260px')
)
fit_model = widgets.Dropdown(
    options=['gaussian', 'lorentzian', 'pseudovoigt'],
    value='lorentzian',
    description='Fit Model:'
)
symmetric_fit = widgets.Checkbox(value=True, description='Symmetric Peak')
use_baseline = widgets.Checkbox(value=True, description='Baseline Correction')
baseline_type = widgets.Dropdown(
    options=['average', 'linear', 'polynomial'],
    value='linear',
    description='Baseline:'
)
smoothing_window = widgets.IntSlider(
    value=11, min=3, max=51, step=2,
    description='Smooth Win:',
    layout=widgets.Layout(width='320px')
)
smoothing_poly = widgets.IntSlider(
    value=3, min=1, max=5, step=1,
    description='Polyorder:',
    layout=widgets.Layout(width='260px')
)
pixel_widget = widgets.IntText(
    value=256,
    description='Pixel:',
    layout=widgets.Layout(width='160px')
)
frames_widget = widgets.IntText(
    value=200,
    description='Frames:',
    layout=widgets.Layout(width='160px')
)

region_labels = [
    '234U', '235U', '238U',
    '234UO', '235UO', '238UO',
    '234UO2', '235UO2', '238UO2'
]
peak_centers = {
    '234U': 234.0, '235U': 235.0, '238U': 238.0,
    '234UO': 250.0, '235UO': 251.0, '238UO': 254.0,
    '234UO2': 266.0, '235UO2': 267.0, '238UO2': 270.0
}

range_width_widgets = {}
baseline_offset_widgets = {}
for label in region_labels:
    center_w = widgets.FloatText(
        value=peak_centers[label],
        description=f'{label}:',
        layout=widgets.Layout(width='170px')
    )
    # 234-series: default ±0.5; others ±1.0
    default_width = 0.5 if label in ['234U', '234UO', '234UO2'] else 1.0
    width_w = widgets.FloatText(
        value=default_width,
        description='±range:',
        layout=widgets.Layout(width='150px')
    )
    bsoff_w = widgets.FloatText(
        value=1.0,
        description='±baseline:',
        layout=widgets.Layout(width='170px')
    )
    range_width_widgets[label] = (center_w, width_w)
    baseline_offset_widgets[label] = bsoff_w

fit_button = widgets.Button(
    description='Fit and Calculate',
    layout=widgets.Layout(width='220px')
)
save_button = widgets.Button(
    description='Save Results',
    layout=widgets.Layout(width='160px')
)

ui = widgets.VBox([
    upload_widget,
    widgets.HBox([fit_model, symmetric_fit]),
    widgets.HBox([use_baseline]),
    widgets.HBox([baseline_type]),
    widgets.HBox([smoothing_window, smoothing_poly]),
    widgets.HBox([pixel_widget, frames_widget]),
    widgets.HTML('<b>Edit Peak Centers and Ranges</b>'),
    *[
        widgets.HBox([
            range_width_widgets[l][0],
            range_width_widgets[l][1],
            baseline_offset_widgets[l]
        ])
        for l in region_labels
    ],
    widgets.HBox([fit_button, save_button])
])

# === [FUNCTIONS] ===

def get_model(model_type, symmetric=True):
    if model_type == 'gaussian':
        return GaussianModel() if symmetric else (
            SkewedGaussianModel() if SkewedGaussianModel else GaussianModel()
        )
    if model_type == 'lorentzian':
        return LorentzianModel() if symmetric else VoigtModel()
    if model_type == 'pseudovoigt':
        return PseudoVoigtModel() if symmetric else VoigtModel()
    raise ValueError('Unsupported model type')


def fit_peak_custom_baseline(
    x_all, y_all, model_type, mz_range,
    baseline_offset, symmetric, use_bs, bs_mode
):
    mask = (x_all >= mz_range[0]) & (x_all <= mz_range[1])
    x_peak = x_all[mask]
    y_peak = y_all[mask]
    if x_peak.size < 5:
        raise ValueError('Peak range too narrow')

    model = get_model(model_type, symmetric)
    c_guess = (mz_range[0] + mz_range[1]) / 2
    h_guess = float(np.max(y_peak))
    s_guess = (mz_range[1] - mz_range[0]) / 4
    a_guess = h_guess * s_guess * np.sqrt(2 * np.pi)
    params = model.make_params(center=c_guess, amplitude=a_guess, sigma=s_guess)
    init_fit = model.fit(y_peak, params, x=x_peak)

    if use_bs:
        if bs_mode == 'average':
            lval = np.interp(c_guess - baseline_offset, x_peak, init_fit.best_fit)
            rval = np.interp(c_guess + baseline_offset, x_peak, init_fit.best_fit)
            baseline = (lval + rval) / 2
            y_corr = y_peak - baseline
        elif bs_mode == 'linear':
            c = c_guess
            lmask = (x_all >= c - baseline_offset) & (x_all < c - baseline_offset / 2)
            rmask = (x_all > c + baseline_offset / 2) & (x_all <= c + baseline_offset)
            xb = np.concatenate([x_all[lmask], x_all[rmask]])
            yb = np.concatenate([y_all[lmask], y_all[rmask]])
            p = np.polyfit(xb, yb, 1)
            y_corr = y_peak - np.polyval(p, x_peak)
        elif bs_mode == 'polynomial':
            p = np.polyfit(x_peak, y_peak, 2)
            y_corr = y_peak - np.polyval(p, x_peak)
        else:
            y_corr = y_peak
    else:
        y_corr = y_peak

    final_fit = model.fit(
        y_corr,
        model.make_params(center=c_guess, amplitude=a_guess, sigma=s_guess),
        x=x_peak
    )
    resid = y_corr - final_fit.best_fit
    ss_res = np.sum(resid ** 2)
    ss_tot = np.sum((y_corr - np.mean(y_corr)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    area = final_fit.params['amplitude'].value
    return final_fit, x_peak, y_peak, y_corr, area, resid, r2


def run_fitting(_):
    global data, x, y, fig, results_last_filename, df_summary
    clear_output(wait=True)
    display(ui)

    if x is None or y is None:
        print('Please upload a .txt file.')
        return

    if smoothing_window.value >= len(y):
        print("Error: Smoothing window too large.")
        return

    y_smooth = savgol_filter(y, smoothing_window.value, smoothing_poly.value)
    ranges = {
        l: (
            range_width_widgets[l][0].value - range_width_widgets[l][1].value,
            range_width_widgets[l][0].value + range_width_widgets[l][1].value
        )
        for l in region_labels
    }
    offsets = {l: baseline_offset_widgets[l].value for l in region_labels}

    fig, axes = plt.subplots(3, 3, figsize=(14, 14))
    axes = axes.flatten()
    areas = {}
    r2s = {}

    for i, l in enumerate(region_labels):
        try:
            res, xfit, yfit, ycorr, area, resid, r2 = fit_peak_custom_baseline(
                x, y_smooth, fit_model.value, ranges[l],
                offsets[l], symmetric_fit.value,
                use_baseline.value, baseline_type.value
            )
            areas[l] = area
            r2s[l] = r2
            ax = axes[i]
            ax.plot(xfit, yfit, label='Smoothed')
            ax.plot(xfit, res.best_fit, 'r--')
            ax.grid(True)
            ax.set_title(f"{l} (R²={r2:.2f})")
        except Exception as e:
            print(f'Error fitting {l}: {e}')
            continue

    plt.tight_layout()
    plt.show()

    rows = []
    for l in region_labels:
        rows.append({
            'Ions': l,
            'Area': areas.get(l, np.nan),
            'R²': r2s.get(l, np.nan),
            '235U Isotopic Ratio': ''
        })

    # 235U ratio including 234 and 238 for each family
    def calc_ratio_3(a234, a235, a238):
        s = a234 + a235 + a238
        return a235 / s if s > 0 else np.nan

    for a234, a235, a238 in [
        ('234U', '235U', '238U'),
        ('234UO', '235UO', '238UO'),
        ('234UO2', '235UO2', '238UO2')
    ]:
        r = calc_ratio_3(
            areas.get(a234, 0),
            areas.get(a235, 0),
            areas.get(a238, 0)
        )
        for row in rows:
            if row['Ions'] == a235:
                row['235U Isotopic Ratio'] = f"{r * 100:.2f}%" if not pd.isna(r) else ''

    df = pd.DataFrame(rows)

    # --- Counts & Area rules ---
    pixel = pixel_widget.value
    frames = frames_widget.value

    df.insert(1, 'Counts', df['Area'] * frames * (pixel ** 2))
    df['Counts'] = pd.to_numeric(df['Counts'], errors='coerce')

    # Non-negative for 234-series
    df.loc[
        df['Ions'].isin(['234U', '234UO', '234UO2']) &
        (df['Counts'] <= 0),
        'Counts'
    ] = 0

    # If Counts == 0 → Area = 0 (numeric, before formatting)
    df.loc[df['Counts'] == 0, 'Area'] = 0

    # Format Area, Counts, R² as strings for display
    df['Area'] = df['Area'].apply(
        lambda v: '' if pd.isna(v) else f"{v:.2f}"
    )
    df['Counts'] = df['Counts'].apply(
        lambda v: '' if pd.isna(v) else f"{v:.2f}"
    )
    df['R²'] = df['R²'].apply(
        lambda v: '' if pd.isna(v) else f"{v:.2f}"
    )

    def sum_counts(names):
        vals = pd.to_numeric(
            df.loc[df['Ions'].isin(names), 'Counts'].replace('', np.nan),
            errors='coerce'
        )
        return float(np.nansum(vals))

    u234 = ['234U', '234UO', '234UO2']
    u235 = ['235U', '235UO', '235UO2']
    u238 = ['238U', '238UO', '238UO2']

    s234 = sum_counts(u234)
    s235 = sum_counts(u235)
    s238 = sum_counts(u238)

    ratio235 = (s235 / (s234 + s235 + s238) * 100) if (s234 + s235 + s238) > 0 else np.nan

    summary = pd.DataFrame([
        {
            'Ions': 'Sum(234U,234UO,234UO2)',
            'Counts': f"{s234:.2f}",
            'R²': '',
            '235U Isotopic Ratio': ''
        },
        {
            'Ions': 'Sum(235U,235UO,235UO2)',
            'Counts': f"{s235:.2f}",
            'R²': '',
            '235U Isotopic Ratio': f"{ratio235:.2f}%"
        },
        {
            'Ions': 'Sum(238U,238UO,238UO2)',
            'Counts': f"{s238:.2f}",
            'R²': '',
            '235U Isotopic Ratio': ''
        }
    ])

    df = pd.concat([df, summary], ignore_index=True)
    df.index = np.arange(1, len(df) + 1)

    display(df)

    df_summary = df.copy()
    if results_last_filename is None:
        results_last_filename = 'output'


def save_results(b):
    global df_summary, fig, results_last_filename
    if df_summary is None:
        print("No results to save. Please run the fitting first.")
        return

    base_name = results_last_filename or 'output'
    try:
        if fig is not None:
            fig.savefig(f"{base_name}_fit.png", dpi=300)
    except Exception as e:
        print(f"Warning: could not save figure: {e}")

    settings = {
        'Fitting Model': fit_model.value,
        'Symmetric Peak': symmetric_fit.value,
        'Baseline Correction': use_baseline.value,
        'Baseline Type': baseline_type.value,
        'Smoothing Window': smoothing_window.value,
        'Smoothing Polyorder': smoothing_poly.value,
        'Pixel': pixel_widget.value,
        'Frames': frames_widget.value
    }

    csv_path = f"{base_name}_summary.csv"
    with open(csv_path, 'w', encoding='utf-8') as f:
        df_summary.to_csv(f, index=False)
        f.write('\nFitting Parameters Used:\n')
        for k, v in settings.items():
            f.write(f" {k}: {v}\n")
        f.write('Peak Regions (m/z range):\n')
        for label in region_labels:
            c = range_width_widgets[label][0].value
            w = range_width_widgets[label][1].value
            o = baseline_offset_widgets[label].value
            f.write(
                f" {label}: center = {c}, ±range = {w}, ±baseline = {o}\n"
            )
    print(f"Saved: {base_name}_summary.csv and {base_name}_fit.png")


def handle_upload(change):
    global data, x, y, results_last_filename
    uploaded = upload_widget.value
    if not uploaded:
        print('No file uploaded.')
        return

    file_info = list(uploaded.values())[0] if isinstance(uploaded, dict) else uploaded[0]
    name = file_info.get('name', 'input.txt')
    content = file_info.get('content', b'')

    df = None
    err = None
    for reader in (
        lambda: pd.read_csv(io.BytesIO(content), sep='\t', comment='#', engine='python'),
        lambda: pd.read_csv(io.BytesIO(content), sep=None, comment='#', engine='python')
    ):
        try:
            df = reader()
            break
        except Exception as e:
            err = str(e)

    if df is None:
        print(f'Failed to parse file: {err}')
        return

    if {x_column, y_column}.issubset(df.columns):
        x_series, y_series = df[x_column], df[y_column]
    else:
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if len(num_cols) >= 2:
            x_series, y_series = df[num_cols[0]], df[num_cols[1]]
            print(f"Warning: using {num_cols[0]} and {num_cols[1]}.")
        else:
            print('Numeric columns missing.')
            return

    valid = (~pd.isna(x_series)) & (~pd.isna(y_series))
    x_vals = np.asarray(x_series[valid].values, float)
    y_vals = np.asarray(y_series[valid].values, float)

    if x_vals.size < 10:
        print(f'Not enough points: {x_vals.size}.')
        return

    data = df
    x = x_vals
    y = y_vals
    results_last_filename = os.path.splitext(name)[0]

    clear_output(wait=True)
    display(ui)
    print(
        f"Loaded: {name} | {len(df)} rows | "
        f"Using columns: X='{x_series.name}', Y='{y_series.name}'"
    )


fit_button.on_click(run_fitting)
save_button.on_click(save_results)
upload_widget.observe(handle_upload, names='value')

display(ui)
